# LAB 07 — Lakeflow Spark Declarative Pipelines

**Goal:** build, run, and verify a Lakeflow Spark Declarative Pipeline end-to-end — wire the RetailHub medallion pipeline in the Databricks UI, write the Lakeflow SQL declarations (ST vs MV, expectations), then verify the result with catalog metadata, row counts, the event log, and the data-quality quarantine.

**Prerequisites:** 07 — Lakeflow Pipelines demo

| Task | What you do |
|---|---|
| Section 1 | Upload the `lakeflow_demo` source files, create and configure the pipeline in the UI, run it and add a new stream file |
| 1 | Complete the Bronze `STREAMING TABLE` declaration with `STREAM read_files(...)` |
| 2 | Complete the Silver declaration with `ON VIOLATION DROP ROW` expectations |
| 3 | Write the Gold declaration for `gold_daily_revenue` |
| 4 | Classify `fact_sales` / `dim_customer` as STREAMING TABLE or MATERIALIZED VIEW and confirm via `information_schema` |
| 5 | Verify row counts: bronze ≥ silver == fact |
| 6 (stretch) | Query the pipeline event log for data-quality metrics |
| 7 (stretch) | Data-quality quarantine: dropped rows = bronze − silver; silver has 0 violations |

> **SOLUTION NOTEBOOK** — full answers for `lab_07_lakeflow_pipeline`.
>
> **Pipeline dependency:** Tasks 1–3 run anywhere. Tasks **4–7 require the Section 1 pipeline to have completed at least one run** on your catalog (target schema `<your_name>_lakeflow`) — run the pipeline from the Lakeflow UI first, then execute those cells.

## Setup

In [ ]:
%run ../setup/00_setup

In [ ]:
# Lakeflow pipeline target schema (matches Step 2 config)
user_name = CATALOG.replace(f"{CATALOG_PREFIX}_", "")
user_schema = f"{user_name}_lakeflow"
print(f"Pipeline target schema: {user_schema}")

## Section 1: Workshop — Building the Pipeline

In this hands-on workshop, we create a complete Lakeflow pipeline from SQL files — uploading source code, configuring the pipeline in the Databricks UI, running it, and validating the results.



### SQL Files Overview

The pipeline source code is organized by medallion layer — each SQL file declares one table or view.

![../../assets/images/training_2026/day3/57bea06e969c45dab4de8bea9ec980b1.webp](../../assets/images/training_2026/day3/57bea06e969c45dab4de8bea9ec980b1.webp)

### Step 1: Upload SQL Files

The pipeline source code is the folder **`materials/lakeflow/lakeflow_demo/transformations/`** of the training repository. It contains three subfolders:
- `01_bronze/` — `bronze_customers.sql`, `bronze_orders.sql`, `bronze_products.sql`
- `02_silver/` — `silver_customers.sql`, `silver_orders.sql`, `silver_products.sql`
- `03_gold/` — `dim_customer.sql`, `dim_date.sql`, `dim_payment_method.sql`, `dim_product.sql`, `dim_store.sql`, `fact_sales.sql`

**Option A: Via UI** — Workspace → Users → your user folder → create a `lakeflow_demo` folder with a `transformations` folder inside, recreate `01_bronze`, `02_silver`, `03_gold` there and upload the SQL files from the matching folders of `materials/lakeflow/lakeflow_demo/transformations/`

**Option B: Via Git Folders** — Git Folders → Add Git Folder → clone the training repository; the source folder is then `<repo>/materials/lakeflow/lakeflow_demo/transformations/` (nothing to upload)

### Step 2: Create Pipeline

Use the catalog/schema printed by the Setup cell (`CATALOG` and `Pipeline target schema`).

1. **Jobs & Pipelines** → **Create** → **ETL pipeline**
2. **Catalog:** `retailhub_<your_name>` (the `CATALOG` printed by Setup)
3. **Pipeline Name:** `lakeflow_pipeline_<your_name>`
4. **Target Schema:** `<your_name>_lakeflow` (the `Pipeline target schema` printed by Setup)
5. **Source Code:** Add existing assets → **Pipeline root folder** = the `lakeflow_demo` folder; **Source code path** = its `transformations/` folder

<img src="../../assets/images/fab4fef8e72d4d5786ba818e6c2f73c5.png" width="800">

*Jobs & Pipelines → Create new → ETL pipeline.*

<img src="../../assets/images/a4e21cb35d3b45f2ba184877139cdc48.png" width="800">

*Give the pipeline its name: `lakeflow_pipeline_<your_name>`.*

<img src="../../assets/images/4e75e96544f142508c3bd8b0b4dbf446.png" width="800">

*If `<your_name>_lakeflow` does not exist yet, create it from the schema selector (Create a new schema).*

![../../assets/images/training_2026/day3/dc9c3eea154e4cb29aee62e6edd5bcca.webp](../../assets/images/training_2026/day3/dc9c3eea154e4cb29aee62e6edd5bcca.webp)

*Add existing assets: root folder = `lakeflow_demo`, source code path = `lakeflow_demo/transformations`.*

### Step 3: Configure Variables

1. Open **Settings** in the pipeline editor and scroll to **Pipeline configuration** (the **Configuration** section).

<img src="../../assets/images/b2fa841a52004939ac2679f9edef8dc3.png" width="800">

*Settings panel — the Configuration section with the Add configuration button.*

2. Click **Add configuration** — a key/value form opens.

<img src="../../assets/images/dc31d5dec7444c1d959b8e1f220c10ce.png" width="800">

*Configuration form — one row per key; the Key field holds only the name (e.g. `product_path`), the path goes into Value.*

3. Enter these keys and values (replace `<your catalog>` with the `CATALOG` printed by Setup):

| Key | Value |
|-----|-------|
| `customer_path` | `/Volumes/<your catalog>/default/datasets/customers` |
| `order_path` | `/Volumes/<your catalog>/default/datasets/orders` |
| `product_path` | `/Volumes/<your catalog>/default/datasets/products/products.parquet/` |

4. Click **Save** — the editor prepares the pipeline graph and the DAG appears.

<img src="../../assets/images/6f16ad4f7fd941ebbb4fb286dbb8fbfd.png" width="800">

*Pipeline editor after saving — "Preparing a new graph…".*

![../../assets/images/training_2026/day3/05f00c9fedb54b0b81ec65bf182a92af.webp](../../assets/images/training_2026/day3/05f00c9fedb54b0b81ec65bf182a92af.webp)

*The finished DAG: bronze → silver → gold (`dim_*` materialized views and the `fact_sales` streaming table).*

### Step 4: Run the Pipeline

Start the pipeline, then test incremental processing by adding a new data file.

![../../assets/images/training_2026/day3/8d06de8a2a674cc1bc119b5d91b2d1ce.webp](../../assets/images/training_2026/day3/8d06de8a2a674cc1bc119b5d91b2d1ce.webp)

*A completed pipeline run with the full DAG.*

1. Click **Start** and wait until the run shows **Completed**.
2. Copy **one** new stream file from `datasets/demo/ingestion/orders/stream/` (`orders_stream_004.json`, `orders_stream_005.json`, `orders_stream_006.json`) into `datasets/orders/stream/` — either download/upload it in **Catalog Explorer** (your catalog → `default` → Volumes → `datasets`), or run in a notebook cell:

```python
dbutils.fs.cp(
    "/Volumes/<your_catalog>/default/datasets/demo/ingestion/orders/stream/orders_stream_004.json",
    "/Volumes/<your_catalog>/default/datasets/orders/stream/orders_stream_004.json",
)
```

3. Click **Start** again — only the new file is processed (check the row counts / **Event Log** of `bronze_orders`).

### Step 5: Verify Results

In [ ]:
# Check fact_sales with joins to dimensions
display(spark.sql(f"""
    SELECT 
        f.order_id,
        c.first_name || ' ' || c.last_name AS customer_name,
        p.product_name,
        d.date,
        f.quantity,
        f.net_amount
    FROM {CATALOG}.{user_schema}.fact_sales f
    LEFT JOIN {CATALOG}.{user_schema}.dim_customer c ON f.customer_key = c.customer_key
    LEFT JOIN {CATALOG}.{user_schema}.dim_product p ON f.product_key = p.product_key
    LEFT JOIN {CATALOG}.{user_schema}.dim_date d ON f.order_date_key = d.date_key
    LIMIT 10
"""))

In [ ]:
# Find customers with change history
display(spark.sql(f"""
    SELECT 
        customer_id, first_name, city,
        __START_AT, __END_AT,
        CASE WHEN __END_AT IS NULL THEN 'Current' ELSE 'Historical' END AS status
    FROM {CATALOG}.{user_schema}.silver_customers
    WHERE customer_id IN (
        SELECT customer_id 
        FROM {CATALOG}.{user_schema}.silver_customers 
        GROUP BY customer_id HAVING COUNT(*) > 1
    )
    ORDER BY customer_id, __START_AT
"""))

### Monitoring and Troubleshooting

Common issues encountered when running Lakeflow pipelines and how to resolve them.

| Issue | Cause | Solution |
|---------|-----------|-------------|
| Pipeline hangs | Compute cannot start | Check serverless compute availability / quota |
| Missing data | Constraint DROP ROW | Check Data Quality tab |
| Schema mismatch | Schema change | Full refresh |

## Section 2: Practice — Lakeflow SQL Declarations

Write and verify Lakeflow SQL syntax for each medallion layer.

> Tasks 1–3 don't need the pipeline — do them while it runs; Tasks 4–7 need a completed run.

### Task 1: Write Bronze Declaration

Complete the SQL below to create a Bronze streaming table from JSON files — replace every `____` placeholder.

**What you need to do:**
1. `CREATE OR REFRESH ____ TABLE` — the keyword that makes the table **incremental**
2. `FROM STREAM ____(...)` — the Lakeflow **file-ingestion function**
3. Inside the function — the source path (`/Volumes/{CATALOG}/default/datasets/orders/stream/`) and `format => 'json'`

**This SQL would go in a pipeline SQL file.** Here we practice the syntax.

In [ ]:
bronze_sql = f"""
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT *
FROM STREAM read_files(
    '/Volumes/{CATALOG}/default/datasets/orders/stream/',
    format => 'json',
    inferColumnTypes => true
);
"""

print("Bronze SQL declaration:")
print(bronze_sql)

#### Hint — Task 1

The goal is to declare a **Bronze streaming table** that ingests raw JSON files using Lakeflow's `read_files` function.

**STREAMING TABLE vs regular table**
A `STREAMING TABLE` processes data incrementally — it tracks what has been read and only picks up new files on subsequent pipeline runs. This is the standard pattern for Bronze ingestion. The `STREAM` keyword before `read_files(...)` enables incremental file processing: Lakeflow creates a checkpoint and on re-run processes only new files, not the entire directory.

**Why `read_files` instead of `spark.read.format(...)`?**
`read_files()` is the Lakeflow-native function for file ingestion. It integrates with Auto Loader, handles schema evolution, and supports all common formats (`json`, `csv`, `parquet`). Unlike `spark.read`, it supports both batch (`read_files`) and streaming (`STREAM read_files`) modes within a Lakeflow pipeline.

In [ ]:
# -- Validation --
assert "____" not in bronze_sql, "Replace every ____ placeholder in bronze_sql"
_sql = " ".join(bronze_sql.upper().split())          # normalise whitespace
assert "CREATE OR REFRESH STREAMING TABLE BRONZE_ORDERS" in _sql, "Should declare a STREAMING TABLE bronze_orders"
assert "STREAM READ_FILES(" in _sql, "Should read incrementally with STREAM read_files(...)"
assert "FORMAT =>" in _sql and "/VOLUMES/" in _sql, "Pass the Volume path and format => 'json' to read_files()"
print("Task 1 OK: Bronze declaration syntax correct")


### Task 2: Write Silver Declaration with Expectations

Complete the Silver layer with data quality constraints.

> **Note:** this is a simplified practice version with 2 constraints; the deployed pipeline declares 5 constraints (see `materials/lakeflow/lakeflow_demo/transformations/02_silver/silver_orders.sql`).

**What you need to do:**
1. Replace the `____` after `ON VIOLATION` in the `valid_id` constraint with `DROP ROW`
2. Do the same for the `positive_amount` constraint

In [ ]:
silver_sql = """
CREATE OR REFRESH STREAMING TABLE silver_orders (
    CONSTRAINT valid_id EXPECT (order_id IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT positive_amount EXPECT (total_price > 0) ON VIOLATION DROP ROW
)
AS SELECT
    order_id,
    customer_id,
    product_id,
    CAST(quantity AS INT) AS quantity,
    CAST(total_price AS DOUBLE) AS total_price,
    CAST(order_date AS DATE) AS order_date,
    payment_method,
    store_id,
    current_timestamp() AS processed_at
FROM STREAM(bronze_orders);
"""

print("Silver SQL declaration:")
print(silver_sql)

#### Hint — Task 2

The goal is to add **data quality expectations** to the Silver layer — the first line of defense against bad data.

**How expectations work**
Expectations are `CONSTRAINT` declarations in the `CREATE` statement. Each constraint has a name, a boolean expression, and a violation action.

| Action | Behavior | When to use |
|--------|----------|-------------|
| `ON VIOLATION DROP ROW` | Row removed silently; violation logged | Business rule violations (null IDs, negative quantities) |
| `ON VIOLATION FAIL UPDATE` | Entire pipeline run fails | Critical integrity constraints — no data is better than bad data |
| *(no action)* | Row kept; violation count logged only | Monitoring without blocking |

**Exam Tip:** `DROP ROW` is the most commonly tested action. Know that it silently discards the row and logs the violation count in the Event Log — it does NOT fail the pipeline.

In [ ]:
# -- Validation --
assert "CONSTRAINT" in silver_sql.upper(), "Should have CONSTRAINT declarations"
assert "DROP ROW" in silver_sql.upper(), "Should use ON VIOLATION DROP ROW"
assert "bronze" in silver_sql.lower(), "Should reference bronze_orders"
print("Task 2 OK: Silver declaration with expectations correct")

### Task 3: Write Gold Declaration

Create a Materialized View for daily revenue summary.

**What you need to do:**
1. Replace the `-- ____ gold_daily_revenue` line with the full `CREATE OR REFRESH MATERIALIZED VIEW gold_daily_revenue` header
2. Keep the `AS SELECT` aggregation below it unchanged

In [ ]:
gold_sql = """
CREATE OR REFRESH MATERIALIZED VIEW gold_daily_revenue
AS SELECT
    order_date,
    SUM(total_price) AS total_revenue,
    COUNT(*) AS total_orders,
    AVG(total_price) AS avg_order_value
FROM silver_orders
GROUP BY order_date
ORDER BY order_date;
"""

print("Gold SQL declaration:")
print(gold_sql)

#### Hint — Task 3

The goal is to write a **Gold Materialized View** — the final aggregated layer optimized for analytics.

**MATERIALIZED VIEW vs STREAMING TABLE**
A `MATERIALIZED VIEW` is recomputed from scratch on each pipeline run — Lakeflow re-reads the entire source and replaces the result. This is correct for aggregations (`SUM`, `COUNT`, `GROUP BY`) where appending would produce wrong totals.

A `STREAMING TABLE` at Gold (like `fact_sales`) processes records incrementally — each batch from Silver adds new rows. This works for fact tables where records are immutable.

**Rule of thumb:** `MATERIALIZED VIEW` for dimension tables and aggregated summaries. `STREAMING TABLE` for fact tables and append-only datasets.

In [ ]:
# -- Validation --
assert "MATERIALIZED VIEW" in gold_sql.upper(), "Gold should use MATERIALIZED VIEW"
assert "silver" in gold_sql.lower(), "Should reference silver_orders"
print("Task 3 OK: Gold Materialized View declaration correct")

### Task 4: Classify STREAMING TABLE vs MATERIALIZED VIEW

Your pipeline (Section 1) created both object types. Prove you can tell them apart **from catalog metadata**, not just from the source SQL.

> **Run after your pipeline completes** — requires the Section 1 pipeline to have completed at least one run.

| Feature | STREAMING TABLE | MATERIALIZED VIEW |
|---------|----------------|-------------------|
| Processing mode | Incremental (append-only) | Full recomputation (or incremental where supported) |
| Best for | Append-only sources (files, CDC) | Aggregations, joins, dimension tables |
| Read from source | `STREAM(table_name)` | `table_name` |
| Supports expectations | Yes | Yes |

**What you need to do:**
1. Fill in the `answer` dict — classify `fact_sales` and `dim_customer`
2. Run the metadata query (provided) and make the validation pass

In [ ]:
answer = {
    "fact_sales":   "STREAMING_TABLE",
    "dim_customer": "MATERIALIZED_VIEW",
}

type_df = spark.sql(f"""
    SELECT table_name, table_type
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema = '{user_schema}'
      AND table_name IN ('fact_sales', 'dim_customer')
""")
actual = {r["table_name"]: r["table_type"] for r in type_df.collect()}
display(type_df)

#### Hint — Task 4

The goal is to solidify the **difference between STREAMING TABLE and MATERIALIZED VIEW** — a core exam concept — and to check it against the catalog.

**Two ways to inspect the object type**
1. `information_schema` (used in this task):
```sql
SELECT table_name, table_type
FROM <catalog>.information_schema.tables
WHERE table_schema = '<pipeline_target_schema>'
```
`table_type` returns `STREAMING_TABLE` or `MATERIALIZED_VIEW` for pipeline objects.

2. `DESCRIBE EXTENDED <table>` — look at the `Type` row.

**How to reason about it**
- `fact_sales` reads `FROM STREAM(silver_orders)` → incremental appends → which object type must the target be?
- `dim_customer` is declared `CREATE OR REFRESH MATERIALIZED VIEW` → recomputed per refresh → the type name follows from the declaration

**Exam Tip:** If a query uses `FROM STREAM(table)`, the target **must** be a STREAMING TABLE. A MATERIALIZED VIEW cannot reference `STREAM(...)`.

In [ ]:
# -- Validation --
assert len(actual) == 2, (
    f"Expected fact_sales and dim_customer in {CATALOG}.{user_schema} — "
    "run the Section 1 pipeline first"
)
assert answer["fact_sales"] == "STREAMING_TABLE", \
    "fact_sales reads FROM STREAM(...) -> it must be a STREAMING_TABLE"
assert answer["dim_customer"] == "MATERIALIZED_VIEW", \
    "dim_customer is CREATE OR REFRESH MATERIALIZED VIEW"
assert answer == actual, f"Catalog disagrees with your answer: {actual}"
print("Task 4 OK: catalog metadata confirms ST vs MV classification")

### Task 5: Verify Pipeline Results

Query the medallion layers your pipeline produced and verify the row-count relationships hold.

> **Run after your pipeline completes** — requires the Section 1 pipeline to have completed at least one run.

**What you need to do:**
1. Count `bronze_orders` with `spark.table(...)` in the pipeline target schema (`{CATALOG}.{user_schema}`) → `bronze_cnt`
2. Count `silver_orders` the same way → `silver_cnt`
3. Count `fact_sales` the same way → `fact_cnt`

In [ ]:
# Requires the pipeline to have run (Section 1)
bronze_cnt = spark.table(f"{CATALOG}.{user_schema}.bronze_orders").count()
silver_cnt = spark.table(f"{CATALOG}.{user_schema}.silver_orders").count()
fact_cnt   = spark.table(f"{CATALOG}.{user_schema}.fact_sales").count()

print(f"bronze_orders : {bronze_cnt:,}")
print(f"silver_orders : {silver_cnt:,}")
print(f"fact_sales    : {fact_cnt:,}")

#### Hint — Task 5

**Where pipeline tables live**
Everything the pipeline declares lands in the Target Schema you configured: `{CATALOG}.{user_schema}.<table>`. They are queryable like any table: `spark.table(f"{CATALOG}.{user_schema}.bronze_orders")`.

**What must hold after a healthy run**
- `bronze_orders` ≥ 100,000 (the batch backfill FLOW) plus any stream files
- `silver_orders` **≤** `bronze_orders` — the five `ON VIOLATION DROP ROW` expectations remove dirty rows
- `fact_sales` == `silver_orders` — each silver order becomes exactly one fact row (left joins to dims never drop rows)

**Bonus check:** the `is_unknown_customer` flag in `fact_sales` marks orders whose `customer_key` fell back to `-1`.

In [ ]:
# -- Validation --
assert bronze_cnt >= 100_000, \
    f"bronze_orders should hold at least the 100k batch backfill, got {bronze_cnt:,}"
assert 0 < silver_cnt <= bronze_cnt, \
    f"silver must be non-empty and <= bronze (expectations DROP ROW): {silver_cnt:,} vs {bronze_cnt:,}"
assert fact_cnt == silver_cnt, \
    f"fact_sales should have one row per silver order: {fact_cnt:,} != {silver_cnt:,}"
print(f"Task 5 OK: bronze={bronze_cnt:,} -> silver={silver_cnt:,} -> fact={fact_cnt:,}")

### Task 6: Query the Pipeline Event Log

> 🏃 **Stretch** — optional in class: do it if you finish early, or after the course.

Read the expectations metrics that Lakeflow recorded for every flow — the audit trail behind the Data Quality tab.

> **Run after your pipeline completes** — requires the Section 1 pipeline to have completed at least one run.

**What you need to do:**
1. Query `event_log(TABLE(...))` for `flow_progress` events of `silver_orders`
2. Parse the `expectations` JSON array and aggregate passed/failed records per constraint

In [ ]:
from pyspark.sql.functions import from_json, explode, col
from pyspark.sql import functions as F

# Requires the pipeline to have run (Section 1)
raw_events = spark.sql(f"""
    SELECT details:flow_progress.data_quality.expectations AS expectations
    FROM event_log(TABLE({CATALOG}.{user_schema}.silver_orders))
    WHERE event_type = 'flow_progress'
      AND details:flow_progress.data_quality.expectations IS NOT NULL
""")

exp_schema = "array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>"
dq_df = (
    raw_events
    .withColumn("exp", explode(from_json(col("expectations"), exp_schema)))
    .groupBy(col("exp.name").alias("name"))
    .agg(
        F.sum("exp.passed_records").alias("passed_records"),
        F.sum("exp.failed_records").alias("failed_records"),
    )
)
display(dq_df)

#### Hint — Task 6

**The `event_log` table function**
Every pipeline writes an event log. For Unity Catalog pipelines, you can access the slice for one table with:
```sql
SELECT * FROM event_log(TABLE(catalog.schema.silver_orders))
```

**JSON path navigation with `:`**
`details:flow_progress.data_quality.expectations` extracts the expectations array (as a JSON string) from the event payload. Filter to `event_type = 'flow_progress'` rows where it is not null.

**Parsing the array in PySpark**
- `from_json(<json string column>, exp_schema)` turns the string into an array of structs (`exp_schema` is provided in the cell)
- `explode(<array>)` gives one row per expectation; fields are then reachable as `exp.name`, `exp.passed_records`, `exp.failed_records`
- Aggregate per constraint with `groupBy(...)` + `F.sum(...)`

**What to look for**
- `failed_records` > 0 on constraints like `valid_quantity` — those are the intentionally dirty rows
- `passed_records` in the millions? You re-ran the pipeline — the log accumulates per flow update, so aggregate with `SUM`.

In [ ]:
# -- Validation --
dq_rows = {r["name"]: r for r in dq_df.collect()}
assert len(dq_rows) >= 3, \
    f"Expected the silver_orders constraints (valid_order_id, valid_quantity, ...), got {list(dq_rows)}"
total_failed = sum(r["failed_records"] for r in dq_rows.values())
total_passed = sum(r["passed_records"] for r in dq_rows.values())
assert total_passed > 0, "Event log should show passed records"
assert total_failed > 0, \
    "The RetailHub feed contains ~3% dirty rows — expected failed_records > 0"
print(f"Task 6 OK: {len(dq_rows)} constraints tracked, "
      f"{total_passed:,} passed / {total_failed:,} failed records logged")

### Task 7: Data-Quality Quarantine — Where Did the Dirty Rows Go?

> 🏃 **Stretch** — optional in class: do it if you finish early, or after the course.

The RetailHub feed intentionally contains **~3% dirty rows** (null keys, zero quantities, missing prices). The Silver expectations use `ON VIOLATION DROP ROW` — those rows never reach Silver. Quantify the quarantine and prove Silver is clean.

> **Run after your pipeline completes** — requires the Section 1 pipeline to have completed at least one run.

**What you need to do:**
1. Compute `dropped_rows` = bronze count − silver count, and the drop percentage
2. Count rows still in **bronze** that violate any Silver constraint (`violations_in_bronze`)
3. Count rows in **silver** that violate any constraint (`violations_in_silver`) — must be **zero**

In [ ]:
VIOLATION_PREDICATE = """
    order_id IS NULL OR customer_id IS NULL OR product_id IS NULL
    OR quantity IS NULL OR quantity = 0
    OR unit_price IS NULL OR unit_price < 0
"""

# Requires the pipeline to have run (Section 1)
dropped_rows = bronze_cnt - silver_cnt
dropped_pct  = dropped_rows / bronze_cnt * 100

violations_in_bronze = (
    spark.table(f"{CATALOG}.{user_schema}.bronze_orders")
    .filter(VIOLATION_PREDICATE).count()
)
violations_in_silver = (
    spark.table(f"{CATALOG}.{user_schema}.silver_orders")
    .filter("""
        order_id IS NULL OR customer_id IS NULL OR product_id IS NULL
        OR quantity IS NULL OR quantity = 0
        OR unit_price IS NULL OR unit_price < 0
    """).count()
)

print(f"Dropped by expectations : {dropped_rows:,} rows ({dropped_pct:.2f}% of bronze)")
print(f"Violations in bronze    : {violations_in_bronze:,}")
print(f"Violations in silver    : {violations_in_silver:,}")

#### Hint — Task 7

**The quarantine equation**
Every bronze row either passes all five expectations into Silver or is dropped:
`bronze_cnt - silver_cnt == rows dropped by expectations`

**Reproducing the constraint predicate in SQL**
The Silver declaration drops a row when ANY of these fail:
```sql
order_id IS NOT NULL
customer_id IS NOT NULL
product_id IS NOT NULL
quantity IS NOT NULL AND quantity <> 0
unit_price IS NOT NULL AND unit_price >= 0
```
So a *violating* row satisfies the negation:
```sql
order_id IS NULL OR customer_id IS NULL OR product_id IS NULL
OR quantity IS NULL OR quantity = 0
OR unit_price IS NULL OR unit_price < 0
```
Count that predicate in bronze (should match `dropped_rows`) and in silver (must be 0).

**Why this matters in production**
`DROP ROW` is silent — without this audit you would never notice a broken upstream feed. Production patterns route violations to a *quarantine table* instead of discarding them (a second flow with the inverted predicate).

In [ ]:
# -- Validation --
assert dropped_rows > 0, "Expected dropped rows — the feed contains ~3% dirty data"
assert 0 < dropped_pct < 10, \
    f"Drop rate should be a few percent (~3%), got {dropped_pct:.2f}% — check your counts"
assert violations_in_bronze == dropped_rows, (
    f"Every dropped row should still be visible in bronze: "
    f"{violations_in_bronze:,} violations vs {dropped_rows:,} dropped"
)
assert violations_in_silver == 0, \
    f"Silver must be clean — found {violations_in_silver} constraint-violating rows"
print(f"Task 7 OK: {dropped_rows:,} dirty rows ({dropped_pct:.2f}%) quarantined out of Silver — Silver is clean")

## Cleanup (Optional)

In [ ]:
# Pipeline cleanup is done via Lakeflow UI (delete the pipeline)
print("LAB 07 complete. Delete the pipeline from Lakeflow UI when done.")

## Lab Complete!

You have:
- Written Bronze STREAMING TABLE declarations
- Written Silver declarations with data quality expectations
- Written Gold MATERIALIZED VIEW declarations
- Classified ST vs MV from `information_schema` metadata
- Verified layer-by-layer row counts after the pipeline run
- Queried the Event Log (`event_log(TABLE(...))`) for expectations metrics
- Audited the quarantine: ~3% dirty rows dropped, Silver provably clean

> **Exam Tip:** In Lakeflow Spark Declarative Pipelines, tables within the same pipeline reference each other directly by name — no prefix needed. Use `STREAM(table_name)` for streaming reads and just `table_name` for batch reads. `ON VIOLATION DROP ROW` silently drops and logs — it does NOT fail the pipeline.

> **Next:** LAB 08 - Lakeflow Jobs & Orchestration

← [07 — Lakeflow Pipelines](../day2/demo/07_lakeflow_pipelines.ipynb) | **[ README](../../README.md)** | [08 — Job Orchestration →](../day3/demo/08_job_orchestration.ipynb)